## Assignment 06: Data Science II

### Setup

In [ ]:
!pip install scikit-learn
!pip install matplotlib
!pip install pandas

In [ ]:
from matplotlib import pyplot as plt

def plot_df_scatter(dataframe, column_x, column_y, colors=None):
    if colors is not None:
        plt.scatter(dataframe[column_x], dataframe[column_y], s=2, c=colors)
    else:
        plt.scatter(dataframe[column_x], dataframe[column_y], s=2)
    plt.xlabel(column_x)
    plt.ylabel(column_y)

In [ ]:
import pandas as pd
df = pd.read_csv("data/machine_data.csv")
df = df.drop(["Timestamp", "Machine"], axis="columns")

plot_df_scatter(df, "Power", "Speed")

### Aufgabe 1: Clustering mit K-Means

Im obigen Setup-Code werden Daten aus der Datei ``data/machine_data.csv`` eingelesen.

Die Datei enthält Betriebsdaten verschiedener Maschinen, die während der Produktion erfasst wurden.
Jeder Eintrag besteht aus einem Zeitstempel, der Maschinennummer und die zu diesem Zeitpunkt an der Maschine gemessenen Leistungsaufnahme ("Power") in kW und die aktuelle Drehzahl ("Speed") in Tausend Umdrehungen pro Minute der Maschine.

Das Ziel dieser Aufgabe ist die erfassende Maschine anhand der gemessenen Werte möglichst präzise zu rekonstruieren, d.h. ein Clustering der Datenpunkte zu den Maschinennummern (Typen) vorzunehmen.
Das Clustering soll anhand der Features "Power" und "Speed" erfolgen, daher werden die Spalten "Timestamp" und die zu rekonstruierende Spalte "Machine" im obigen Code zunächst aus den Daten entfernt.


Im Setup-Code wurde auch bereits eine Visualisierungsfunktion ``plot_df_scatter`` definiert und die Funktion genutzt, um den Datensatz zu visualisieren.

Für den menschlichen Betrachter scheint eine Cluster-Bildung der Datenpunkte intuitiv einfach möglich zu sein.
Nicht ganz so offensichtlich ist jedoch, wie genau ein zugehöriger Algorithmus für automatisiertes Clustering aussieht.

Im Folgenden implementieren wir dazu den in der Vorlesung vorgestellten Algorithmus *K-Means*. 

Zur Erinnerung:

<img style="padding-left:25%; padding-right:25%" width=50% src="figures/K-Means.png"/>

#### 1 A) Distanzfunktion

Wir beginnen mit der Implementierung einer geeigneten Distanzfunktion.
Diese wird in Schritt 2) des Algorithmus benötigt, um zu jedem der Punkte den Abstand zu den jeweiligen Clusterzentren bestimmen zu können.

Wir wählen die euklidische Norm im 2D-Raum als Distanzfunktion. Für zwei Punkte $(x_1, y_1), (x_2, y_2) \in  \mathbb{R}^2$ lässt sich diese Distanz mit Hilfe folgender Formel berechnen:

$|(x_1, y_1) - (x_2, y_2)|_2 = \sqrt{|x_1-x_2|^2 + |y_1-y_2|^2}$

Implementiere die Distanz-Funktion ``euklidean_distance_2d(x_1, y_1, x_2, y_2)`` so, dass die euklidische Distanz zwischen den beiden Punkten zurückgegeben wird. 

In [ ]:
import math

def euklidean_distance_2d(x_1, y_1, x_2, y_2):
    ### BEGIN SOLUTION
    dist_squared = (x_1 - x_2)**2 + (y_1 - y_2)**2
    return math.sqrt(dist_squared)
    ### END SOLUTION

In [ ]:
assert euklidean_distance_2d(0, 0, 0, 0) == 0
assert euklidean_distance_2d(1, 0, 0, 1) == euklidean_distance_2d(0, -1, 1, 0)
assert math.isclose(euklidean_distance_2d(1.3, -1.5, -0.5, -2.2), 1.93132, abs_tol=1e-4)

#### 1 B) Cluster Assignment

Nun vervollständigen wir die Implementierung von Schritt 2 des *K-Means*-Algorithmus.
Dazu implementieren wir die Funktionen ``assign_to_cluster(x, y, cluster_indices, cluster_centers)`` und ``assign_clusters(data_x, data_y, cluster_indices, cluster_centers)``.

*Hinweis:* Folgende Parameternamen und -beschreibungen sind für das gesamte Übungsblatt relevant und werden auch in den folgenden Teilaufgaben gleichwertig verwendet:
* ``cluster_indices``: Eine duplikatfreie Liste mit ganzzahligen Indizes (Identifiern) der zu optimierenden Cluster, z.B. ``[1,2,3]``. (Die genauen Werte haben dabei keine Relevanz.)
* ``cluster_centers``: Eine Liste der Clusterzentren (Schwerpunkte). Die Länge der Liste entspricht der Länge der Liste ``cluster_indices``. Jeder Eintrag ist ein Tupel ``(x, y)``, welches die x- und y-Koordinate des Clustermittelpunkts angibt.
* ``data_x``: Eine Liste der x-Koordinaten aller Datenpunkte.
* ``data_y``: Eine Liste der y-Koordinaten aller Datenpunkte.

Gehe bei der Implementierung der beiden Funktionen wiefolgt vor:

1. Die Funktion ``assign_to_cluster(x, y, cluster_indices, cluster_centers)`` erhält als Argument die x- und y-Koordinate eines beliebigen Datenpunktes, sowie die Liste aller Clusterzentren ``cluster_centers``.
Die Funktion soll den Index des Clusters (entsprechend der Liste ``cluster_indices``) mit minimaler Distanz zum vorliegenden Datenpunkt (``x``, ``y``) zurückgeben.
Verwende die zuvor implementierte Funktion ``euklidean_distance_2d`` zur Distanzberechnung.

2. Die Funktion ``assign_clusters(data_x, data_y, cluster_indices, cluster_centers)`` enthält in den Paramtern ``data_x`` und ``data_y`` die Liste der x- und y-Koordinaten aller Datenpunkte des Datensatzes.
Implementiere die Funktion so, dass eine Liste ``cluster_assignments`` zurückgegeben wird, die für jeden Punkt des Datensatzes den Index des nächstliegenden Clusterzentrums enthält. (Der ``i``-te Eintrag der Liste soll den Index des nächsten Clusterzentrums für den Datenpunkt ``data_x[i], data_y[i]`` enthalten.)

In [ ]:
def assign_to_cluster(x, y, cluster_indices, cluster_centers):
    min_index = -1
    min_distance = 1e6
    for i, cluster_index in enumerate(cluster_indices):
        center_x, center_y = cluster_centers[i]
        ### BEGIN SOLUTION
        cluster_distance = euklidean_distance_2d(x, y, center_x, center_y)
        if cluster_distance < min_distance:
            min_distance = cluster_distance
            min_index = cluster_index
        ### END SOLUTION
    return min_index

def assign_clusters(data_x, data_y, cluster_indices, cluster_centers):
    cluster_assignments = []
    ### BEGIN SOLUTION
    for x, y in zip(data_x, data_y):
        cluster_index = assign_to_cluster(x, y, cluster_indices, cluster_centers)
        cluster_assignments.append(cluster_index)
    ### END SOLUTION
    return cluster_assignments

In [ ]:
_indices = [1, 2, 3]
cluster_centers = [(1, 1), (3.1, 3.1), (5, 5)]
assert assign_to_cluster(1, 1, _indices, cluster_centers) == 1
assert assign_to_cluster(2, 2, _indices, cluster_centers) == 1
assert assign_to_cluster(3, 3, _indices, cluster_centers) == 2
assert assign_to_cluster(100, 100, _indices, cluster_centers) == 3
assert assign_to_cluster(100, 100, [3, 2, 1], cluster_centers) == 1
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_assignments = assign_clusters(test_x, test_y, _indices, cluster_centers)
assert _assignments[0] == 1
assert _assignments[1] == 1
assert _assignments[2] == 2
assert _assignments[3] == 3
assert _assignments[4] == 3

#### 1 C) Berechnung der Cluster-Zentren (Schwerpunkte)

In dieser Teilaufgabe wird Schritt 3) des *K-Means*-Algorithmus, d.h. die (Neu-)Berechnung der Clusterzentren betrachtet.

Um diesen Schritt zu implementieren, soll die Funktion ``compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments)`` implementiert werden.
Die Parameter entsprechen den zuvor beschriebenen (siehe vorherige Teilaufgabe).
Zurückgegeben werden soll eine Liste ``new_cluster_centers``, welche für jedes Cluster den neu berechneten Cluster-Schwerpunkt (Clusterzentrum) zurückgibt.

Zur Erinnerung: Der Schwerpunkt einer Menge von Punkten $(x, y)_i, i=1,...,n \in \mathbb{R}^2$ lässt sich berechnen als $(\sum_{i=1}^n \frac{x_i}{n}, \sum_{i=1}^n \frac{y_i}{n})$.

Gehe bei der Implementierung wiefolgt vor:
1. Für jeden Cluster, füge zunächst die x- und y-Koordinaten aller Datenpunkte, die diesem Cluster zugewiesen sind den Listen ``cluster_xs`` and ``cluster_ys`` hinzu.
2. Berechne die x- und y-Koordinaten des Cluster-Schwerpunkts und speichere diese in den Variablen ``center_x`` und ``center_y``.

In [ ]:
def compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments):
    new_cluster_centers = []
    for c in cluster_indices:
        cluster_xs = []
        cluster_ys = []
        ### 1. Add x- and y- coordinates of all the cluster's data points to cluster_xs and cluster_ys, respectively.
        ### BEGIN SOLUTION
        for i, (x, y) in enumerate(zip(data_x, data_y)):
            if cluster_assignments[i] == c:
                cluster_xs.append(x)
                cluster_ys.append(y)
        ### END SOLUTION
        ### 2. Compute cluster center's x- and y- coordinates as center_x and center_y
        ### BEGIN SOLUTION
        center_x = sum(cluster_xs) / len(cluster_xs)
        center_y = sum(cluster_ys) / len(cluster_ys)
        ### END SOLUTION
        new_cluster_centers.append((center_x, center_y))
    return new_cluster_centers

In [ ]:
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_indices = [1, 2, 3]
_assignments = [1, 2, 3, 1, 2]
_centers = compute_cluster_centers(test_x, test_y, _indices, _assignments)
assert len(_centers) == len(_indices)
assert _centers[0][0] == 5/2 and _centers[0][1] == 6/2
assert _centers[1][0] == 7/2 and _centers[1][1] == 8/2
assert _centers[2][0] == 3 and _centers[2][1] == 3

#### 1 D) K-Means Algorithmus

Nun können die implementierten Teilschritte zu dem iterativen Vorgehen zur Cluster-(Neu-)Bestimmung des *K-Means*-Algorithmus zusammengesetzt werden.

Dazu wird die Funktion ``k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number)`` implementiert.
Die Parameter ``data_x, data_y, cluster_indices, cluster_centers`` enthalten dabei die oben beschriebenen Daten.
Der zusätzliche Parameter ``iteration_number`` gibt an wie viele Durchläufe der Cluster-Verbesserung durchgeführt werden sollen bevor ein Abbruch der Schleife stattfinden soll.

Der Algorithmus soll dabei vorgehen wie in der Vorlesung beschrieben (siehe Abbildung oben).
Konkret soll also bei ``i`` Iterationen:
* Schritt 3) (Berechnung der Cluster-Schwerpunkte) genau ``i`` mal ausgeführt werden
* Schritt 2) (Cluster-Zuordnung) genau ``i+1`` mal durchgeführt werden

Rückgabe der Funktion sind sowohl die Liste der Clusterzentren ``cluster_centers`` als auch die Liste der Zuweisungen aller Datenpunkte zu den Cluster Indizes ``cluster_assignments``.

In [ ]:
def k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number):
    ### BEGIN SOLUTION
    for i in range(iteration_number):
        cluster_assignments = assign_clusters(data_x, data_y, cluster_indices, cluster_centers)
        cluster_centers = compute_cluster_centers(data_x, data_y, cluster_indices, cluster_assignments)

    cluster_assignments = assign_clusters(data_x, data_y, cluster_indices, cluster_centers)
    
    return cluster_centers, cluster_assignments
    ### END SOLUTION

In [ ]:
cluster_indices = [1, 2, 3]
cluster_centers = [(1, 1), (3.1, 3.1), (5, 5)]
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_centers, _assignments = k_means_clustering(test_x, test_y, cluster_indices, cluster_centers, 0)
assert _centers[0][0] == cluster_centers[0][0] and _centers[0][1] == cluster_centers[0][1]
assert _centers[1][0] == cluster_centers[1][0] and _centers[1][1] == cluster_centers[1][1]
assert _centers[2][0] == cluster_centers[2][0] and _centers[2][1] == cluster_centers[2][1]
assert _assignments[0] == 1
assert _assignments[1] == 1
assert _assignments[2] == 2
assert _assignments[3] == 3
assert _assignments[4] == 3
_centers, _assignments = k_means_clustering(test_x, test_y, cluster_indices, cluster_centers, 1)
assert _centers[0][0] == _centers[0][1] == 1.5
assert _centers[1][0] == _centers[1][1] == 3
assert _centers[2][0] == 4.5
assert _centers[2][1] == 5.5

#### 1 E) Zufällige Initialisierung

Um den implementieren Algorithmus ausführen zu können, ist eine Initialisierung der Cluster erforderlich.
Diese beinhaltet sowohl die Anzahl der Cluster, als auch die Wahl der initialen Clusterzentren.

Dazu implementieren wir eine randomisierte Auswahl von Clusterzentren aus der Menge aller Datenpunkte in der Funktion ``pick_random_cluster_centers(data_x, data_y, cluster_number)``.
Vervollständige die Funktion so, dass die Liste der Clusterzentren ``cluster_centers`` so initialisiert und befüllt wird,
dass Sie die Datenpunkte mit den zufällig ausgewählten Indizes aus ``initial_center_indices`` enthält.

Verwende die bisher fertiggestellten Funktionen anschließend, um in der Funktion ``k_means_random_init(data, cluster_number, iteration_number)`` den vollständigen *K-Means*-Algorithmus mit Zufallsinitialisierung der Clusterzentren zu vereinen.
Die Argumente der Funktion beinhalten dabei neben dem Datensatz (``data``) die Anzahl der Cluster, in welche die Daten unterteilt werden sollen, sowie die Anzahl der durchzuführenden Iterationen.

Innerhalb der Funktion müssen also:
* x- und y-Spalte aus den Gesamt-Datensatz ``data`` extrahiert werden; dabei ist ``data`` ein Array mit Dimension ``(n, 2)``, wobei ``n`` der Anzahl der Datenpunkte im Datensatz entspricht
* die Indizes der Cluster (z.B. ``[1, 2, ..., cluster_number]``) gewählt und in die Liste ``cluster_indices`` geschrieben werden
* eine Initialisierung der Clusterzentren mit Hilfe der Funktion ``pick_random_cluster_centers`` gefunden und in der Variablen ``cluster_centers`` gespeichert werden

Die Rückgaben der Funktion sind, wie oben, die Liste der Clusterzentren ``cluster_centers``, sowie die Liste der Zuweisungen aller Datenpunkte zu den Cluster Indizes ``cluster_assignments``.

In [ ]:
import random

def pick_random_cluster_centers(data_x, data_y, cluster_number):
    data_indices = list(range(len(data_x)))
    random.shuffle(data_indices)
    initial_center_indices = data_indices[:cluster_number]
    ### BEGIN SOLUTION
    cluster_centers = [(data_x[index], data_y[index]) for index in initial_center_indices]
    ### END SOLUTION
    return cluster_centers

def k_means_random_init(data, cluster_number, iteration_number):
    ### BEGIN SOLUTION
    data_x = data[:, 0]
    data_y = data[:, 1]
    cluster_indices = list(range(cluster_number))
    cluster_centers = pick_random_cluster_centers(data_x, data_y, cluster_number)
    ### END SOLUTION
    return k_means_clustering(data_x, data_y, cluster_indices, cluster_centers, iteration_number)

In [ ]:
random.seed(10)
test_x = [1, 2, 3, 4, 5]
test_y = [1, 2, 3, 5, 6]
_centers = pick_random_cluster_centers(test_x, test_y, 2)
assert _centers[0][0] == 4
assert _centers[0][1] == 5
assert _centers[1][0] == 3
assert _centers[1][1] == 3
_centers, _assignments = k_means_random_init(df.values, 2, 2)

#### 1 F) Anwendung und Visualisierung

Nun soll der fertig implementierte *K-Means* Algorithmus auf den zu Beginn eingelesenen Datensatz angewendet und die Ergebnisse visualisiert werden.

Dazu muss die Funktion ``k_means_random_init`` aufgerufen werden. 
Wählen Sie hierbei die Cluster-Anzahl dem Datensatz entsprechend sinnvoll und führen Sie zunächst 10 Iterationen des Algorithmus durch.
Speichern Sie die Ergebnisse des Clusterings in den Variablen ``cluster_centers`` und ``cluster_assignment``.

Die Ergebnisse des Clusterings sollen anschließend mit Hilfe der Funktion ``plot_df_scatter`` visualisiert werden.

**Achtung:**
Sollten die vorherigen Aufgabenteile nicht (korrekt) gelöst worden sein, so kann die nächste Codezelle ausgeführt werden, um den Clustering-Algorithmus mit der im Paket *sklearn* bereits implementierten Variante zu ersetzen.

Führe den Algorithmus mit zufälliger Cluster-Initialisierung mehrfach aus.
Was fällt auf?

In [ ]:
### OPTIONAL: ONLY RUN THIS CODE CELL IF YOU DID NOT SUCCESSFULLY IMPLEMENT THE K-MEANS ALGORITHM YOURSELF
from sklearn import cluster

def k_means_random_init(data, cluster_number, iteration_number):
    kmeans_model = cluster.KMeans(n_clusters=cluster_number, n_init=1, init="random", max_iter=iteration_number)
    cluster_predictions = kmeans_model.fit_predict(data)

    return kmeans_model.cluster_centers_, cluster_predictions

In [ ]:
### BEGIN SOLUTION
cluster_centers, cluster_assignment = k_means_random_init(df.values, 4, 10)
plot_df_scatter(df, "Power", "Speed", cluster_assignment)
### END SOLUTION

In [ ]:
assert len(cluster_centers) == 4
assert len(cluster_assignment) == len(df.values)
for center in cluster_centers:
    assert len(center) == 2

#### Bonus: Intelligente Initialisierung

In dieser Aufgabe könnt Ihr Euch selbst eine "schlauere" Initialisierungs-Idee (Heuristik) überlegen und diese implementieren.

Ziel dabei ist es natürlich eine Methode zu finden, bei der gute Clustering-Ergebnisse nicht durch einen guten Random Seed bedingt werden.

Implementiere dazu die Methode ``initialize_clusters_clever(data, n_clusters, random_state)``. Die Parameter der Methode sind:
* ``data``: Die zu clusternden Daten in einem Array der Dimensionsn ``(n, 2)``
* ``n_clusters``: Die Anzahl der zu optimierenden Cluster
* ``random_state``: Ein spezifisches Objekt zur Initialisierung des Zufallsfaktors, um Reproduzierbarkeit zu gewährleisten (kann insbesondere bei deterministischen Implementierungen ignoriert werden)

Zurückgegeben werden soll wie zuvor eine Liste der gewählten Clusterzentren (Tupel) mit Länge ``n_clusters`` oder ein Array der Dimension ``(n_clusters, 2)``.

Dabei soll natürlich eine möglichst allgemeingültige Implementierung gefunden werden, d.h. nicht mit absoluten Zahlenwerden, die auf den vorliegenden Datensatz getuned sind, gearbeitet werden!

Teste anschließend die Implementierung, indem Du den K-Means Algorithmus mit der neuen Initialisierung ausführst und die Ergebnisse manuell analysierst (im Plot).

*Hinweis*: Eine häufig verwendete Initialisierung-Strategie ist K-means++. Dabei wird ein erstes Cluster-Zentrum zufällig aus den Datenpunkten gewählt. Anschließend werden weitere Cluster-Zentren aus den übrigen Datenpunkten gezogen, dabei ist die Wahrscheinlichkeit einen Datenpunkt zu ziehen proportional zur Summe der Entfernung dieses Datenpunkts zu den bereits gewählten Cluster-Zentren.

In [ ]:
def initialize_clusters_clever(data, n_clusters, random_state):
    random.seed(hash(random_state))
    cluster_centers = [(0, 0) for index in range(n_clusters)]

    ### BEGIN SOLUTION
    # Select the first cluster center randomly
    cluster_center_indices = [random.randint(0, data.shape[0])]
    
    while len(cluster_center_indices) < n_clusters:

        # Compute the sum of distances to cluster centers for all data points (which are not yet cluster centers)
        point_distances = []
        for i in range(len(data)):
            point_cluster_distance_sum = 0
            if i not in cluster_center_indices:
                for j in cluster_center_indices:
                    point_cluster_distance_sum += euklidean_distance_2d(data[i,0], data[i,1], data[j,0], data[j,1])
            point_distances.append(point_cluster_distance_sum)

        # Compute probability distribution for next cluster center choice proportional to distances
        all_distances_sum = sum(point_distances)
        point_probs = [dist / sum(point_distances) for dist in point_distances]

        # Choose the next cluster center according to probability distribution
        random_number = random.random()
        prob_sum = 0
        i = 0
        while random_number >= prob_sum:
            prob_sum += point_probs[i]
            i += 1

        cluster_center_indices.append(i-1)
    
    cluster_centers = [data[i] for i in cluster_center_indices]
    ### END SOLUTION
    return cluster_centers

In [ ]:
def k_means_clever_init(data, cluster_number, iteration_number):
    kmeans_model = cluster.KMeans(n_clusters=cluster_number, n_init=1, init=initialize_clusters_clever, max_iter=iteration_number)
    cluster_predictions = kmeans_model.fit_predict(data)

    return kmeans_model.cluster_centers_, cluster_predictions

cluster_centers, cluster_assignment = k_means_clever_init(df.values, 4, 1)
plot_df_scatter(df, "Power", "Speed", cluster_assignment)